# Hardware-free collect/drop experiment

This notebook runs the real `Navigator` route logic with a `FakeRobot`. The fake records the actions that collection and drop-off are expected to request: reset, approach, lower/lift, claw close/open, and state changes. It does not import or move hardware.

This checks the navigation-to-robot call flow and the fake robot's expected sequence. It does not execute `Robot.Collect_Box()` or `Robot.Drop_Box()` from `robot.py`; those methods depend on the board runtime and hardware-facing classes.

In [1]:
from Navigation import Navigator, strt, uer6, ler6, uwr6, lwr6

class FakeServo:
    def __init__(self, name, events):
        self.name = name
        self.events = events

    def set_angle(self, angle):
        self.events.append((self.name + "_angle", angle))

class FakeRobot:
    """Simulate the robot methods Navigator calls; record all actions."""
    forklift_lower_angle = 0
    forklift_default_angle = 108
    forklift_raise_angle = 200
    claw_close_angle = 50
    claw_open_angle = 110

    def __init__(self, box_polarity=-1, rack_occupied=False):
        self.events = []
        self.box_polarity = 0
        self.has_box = False
        self.detected_polarity = box_polarity
        self.rack_occupied = rack_occupied
        self.forklift_servo = FakeServo("forklift", self.events)
        self.claw_servo = FakeServo("claw", self.events)

    def reset(self):
        self.events.append(("reset",))

    def move_to_box(self, speed):
        self.events.append(("move_to_box", speed))
        return self.detected_polarity

    def Collect_Box(self):
        self.reset()
        self.events.append(("wait_before_pickup", 1000))
        polarity = self.move_to_box(50)
        self.forklift_servo.set_angle(self.forklift_lower_angle)
        self.events.append(("wait_for_forklift_lower", 1000))
        self.claw_servo.set_angle(self.claw_close_angle)
        self.events.append(("wait_for_claw_close", 1000))
        self.forklift_servo.set_angle(self.forklift_default_angle)
        self.box_polarity = polarity
        self.has_box = polarity in (-1, 1)
        self.events.append(("box_state", self.has_box, self.box_polarity))
        return polarity

    def Turn(self, direction):
        self.events.append(("turn", direction))

    def Reverse_Turn(self, direction):
        self.events.append(("reverse_turn", direction))

    def One_Eighty_Degree_Turn(self):
        self.events.append(("turn_180",))

    def Forward_Until_Node(self, speed):
        self.events.append(("forward_to_node", speed))

    def Forward_Until_Rack(self, speed, time=None):
        self.events.append(("forward_to_rack", speed, time))
        return self.rack_occupied

    def drive(self, direction, speed):
        self.events.append(("drive", direction, speed))

    def _set_box_polarity_leds(self, polarity):
        self.events.append(("polarity_leds", polarity))

    def Drop_Box(self):
        self.events.append(("wait_before_drop", 0))
        self.drive(1, 0)
        self.forklift_servo.set_angle(self.forklift_raise_angle)
        self.events.append(("wait_for_forklift_raise", 1000))
        self.claw_servo.set_angle(self.claw_open_angle)
        self.events.append(("wait_for_claw_open", 1000))
        self.has_box = False
        self.box_polarity = 0
        self._set_box_polarity_leds(0)
        self.reset()

print("Loaded Navigator and fake robot. No hardware is accessed.")

Loaded Navigator and fake robot. No hardware is accessed.


In [2]:
# Simulate collecting a south-polarity box.
fake_robot = FakeRobot(box_polarity=-1)
navigator = Navigator(robot=fake_robot, strt=strt)
box = navigator.get_route_for_box_state(has_box=False)
box_route = list(navigator.route)
navigator.FollowRoute()

assert fake_robot.has_box is True
assert fake_robot.box_polarity == -1
assert ("forklift_angle", fake_robot.forklift_lower_angle) in fake_robot.events
assert ("claw_angle", fake_robot.claw_close_angle) in fake_robot.events
assert ("forklift_angle", fake_robot.forklift_default_angle) in fake_robot.events
assert navigator.current is box_route[-2]
print("Pickup sequence recorded; polarity:", navigator.polarity)
print("Actions:", fake_robot.events)

Pickup sequence recorded; polarity: -1
Actions: [('turn_180',), ('forward_to_node', 100), ('forward_to_node', 100), ('turn', 1), ('forward_to_node', 100), ('forward_to_node', 100), ('turn', 1), ('reset',), ('wait_before_pickup', 1000), ('move_to_box', 50), ('forklift_angle', 0), ('wait_for_forklift_lower', 1000), ('claw_angle', 50), ('wait_for_claw_close', 1000), ('forklift_angle', 108), ('box_state', True, -1), ('reverse_turn', -1)]


In [3]:
# Route to the matching rack and simulate a successful drop-off.
rack = navigator.get_route_for_box_state(has_box=True)
rack_route = list(navigator.route)
expected_racks = {
    (1, 1): uer6, (1, -1): ler6,
    (-1, 1): uwr6, (-1, -1): lwr6,
}
assert rack is expected_racks[(box.side, navigator.polarity)]
navigator.FollowRoute()

assert ("claw_angle", fake_robot.claw_open_angle) in fake_robot.events
assert ("forklift_angle", fake_robot.forklift_raise_angle) in fake_robot.events
assert fake_robot.has_box is False
assert fake_robot.box_polarity == 0
assert fake_robot.events.count(("reset",)) >= 2
assert navigator.current is rack_route[-2]
print("Drop-off sequence recorded at rack:", rack.name)
print("Final fake robot state: has_box =", fake_robot.has_box, 
      ", polarity =", fake_robot.box_polarity)
print("All simulated collect/drop assertions passed.")

Drop-off sequence recorded at rack: lwr6
Final fake robot state: has_box = False , polarity = 0
All simulated collect/drop assertions passed.
